# Eksperyment: Wielo-sezonowy trening + uczciwy test boostingu

## Cel
Dotychczasowa architektura trenowała tylko na roku docelowym (~3500 próbek). Wcześniejszy test na pojedynczym sezonie (notebook HGB) pokazał,
że HistGradientBoosting nie bije Random Forest -- ale na tak małej próbie boosting nie ma jak
rozwinąć swojej przewagi. Tutaj zmieniamy architekturę: trenujemy na wielu sezonach
(2001-2023, ~128 tys. próbek po symetryzacji -- praktycznie całe ATP od 2000), walidujemy na 2024 i
testujemy na całym sezonie 2025. To właściwy test hipotezy *"więcej danych => boosting wreszcie
opłacalny"*.

## Metoda (leakage-safe, ten sam matrix dla wszystkich modeli)
- Cechy IDENTYCZNE z baseline (40 cech) -- reużywamy `add_dynamic_features` / `symmetrize_data` z
  `tennis_model.py` przez namespace. Jedyne zmienne to ilość danych treningowych i algorytm.
- **Rozgrzewka cech: sezon 2000** -- liczy historię (forma, H2H, surface form) dla pierwszych meczów
  2001, ale nie wchodzi do zbioru treningowego.
- **Split po roku PLIKU** (`season`), nie po `tourney_date.dt.year`: plik sezonu 2025 zaczyna się od
  United Cup z końca grudnia 2024, więc sama data myli sezon.
- **Label encoding** fitowany TYLKO na treningu (`season < 2024`) -- zero wglądu w val/test.
- **CV chronologiczne** (`TimeSeriesSplit`) + tuning po `neg_log_loss`, ten sam `random_state=42`.
- Trzy modele -- **RF vs HistGradientBoosting vs XGBoost** -- na dokładnie tej samej macierzy
  cech, porównanie match accuracy oraz jakości kalibracji (Brier / log-loss / ECE).

In [ ]:
import sys
from pathlib import Path
import os

_REPO = "https://github.com/StanislawKarwala/TennisPredictionModel.git"
if "google.colab" in sys.modules and not Path("../src/tennis_model.py").exists():
    import subprocess
    subprocess.run(["pip", "install", "-q", "xgboost"])
    subprocess.run(["git", "clone", "-q", _REPO, "/content/tenis"])
    os.chdir("/content/tenis/notebooks")
sys.path.insert(0, str(Path("../src").resolve()))

## 1. Reuse baseline -- pobieramy funkcje feature-engineering
Najpierw ustawiamy zakres treningu (od 2001, rozgrzewka 2000) przez zmienne środowiskowe -- moduł
czyta je przy imporcie. Potem uruchamiamy `tennis_model.py` raz (z wyciszonym outputem) i wyciągamy z
jego namespace funkcje: budowanie cech dynamicznych, symetryzacja, symetryczna metryka match-level
oraz ocena kalibracji. Bierzemy też listę **40 cech** i `cols_base` -- macierz jest taka sama jak w
baseline, tylko zbudowana na wielu sezonach.

In [ ]:
import os
# Trening od 2001 (~128 tys. probek), sezon 2000 jako rozgrzewka cech -- pelny zakres danych od 2000.
# Ustawiamy PRZED importem modulu (czyta env przy imporcie).
os.environ["TENNIS_WARMUP_START"] = "2000"
os.environ["TENNIS_TRAIN_START"] = "2001"

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder

import tennis_model_multiseason as m

WARMUP_START = m.WARMUP_START
TRAIN_START  = m.TRAIN_START
VAL_YEAR     = m.VAL_YEAR
TEST_YEAR    = m.TEST_YEAR
RANDOM_STATE = m.RANDOM_STATE

print("Uruchamiam baseline raz (pobranie funkcji feature-engineering)...")
ns = m.run_baseline_quietly()                      # runpy tennis_model.py (cicho)
add_dynamic_features = ns["add_dynamic_features"]
symmetrize_data = ns["symmetrize_data"]
compute_symmetric_match_evaluation = ns["compute_symmetric_match_evaluation"]
evaluate_calibration_quality = ns["evaluate_calibration_quality"]
features = list(ns["features"])
cols_base = list(ns["cols_base"])
ROUND_ORDER = ns["ROUND_ORDER"]

print(f"\nTrening {TRAIN_START}-{VAL_YEAR-1} | walidacja {VAL_YEAR} | test {TEST_YEAR}")
warmup_desc = f"{WARMUP_START}-{TRAIN_START-1}" if TRAIN_START > WARMUP_START else "BRAK"
print(f"Rozgrzewka cech: {warmup_desc}")
print(f"Liczba cech (identyczna z baseline): {len(features)}")
print(f"XGBoost dostepny: {m.HAS_XGB}")

Uruchamiam baseline raz (pobranie funkcji feature-engineering)...



Trening 2001-2023 | walidacja 2024 | test 2025
Rozgrzewka cech: 2000-2000
Liczba cech (identyczna z baseline): 40
XGBoost dostepny: True


## 2. Wczytujemy wszystkie sezony 2000-2025 i dzielimy na rozgrzewkę / span
`load_years` czyta pliki `atp_matches_<rok>.csv`, parsuje `tourney_date`, sortuje chronologicznie i
oznacza `season` rokiem pliku. Robimy jedno wczytanie 2000..2025, a potem dzielimy:
- `historical` = sezony przed 2001 (czyli sezon 2000 -- tylko rozgrzewka cech, nie trafia do treningu),
- `span` = 2001..2025 (na nich liczymy cechy dynamiczne i które potem splitujemy).

In [ ]:
full = m.load_years(range(WARMUP_START, TEST_YEAR + 1), cols_base)
full = m.add_static_features(full, ROUND_ORDER)
historical = full[full["season"] < TRAIN_START].reset_index(drop=True)
span = full[full["season"] >= TRAIN_START].reset_index(drop=True)

print(f"Wczytano łącznie {len(full)} meczów ({WARMUP_START}-{TEST_YEAR})")
print(f"  rozgrzewka (<{TRAIN_START}): {len(historical)} meczów")
print(f"  span      (>={TRAIN_START}): {len(span)} meczów")

# Ile meczow per sezon w span (kontrola spojnosci danych)
print("\nMecze per sezon (span):")
print(span["season"].value_counts().sort_index().to_string())

Wczytano łącznie 70021 meczów (2000-2025)
  rozgrzewka (<2001): 2868 meczów
  span      (>=2001): 67153 meczów

Mecze per sezon (span):
season
2001    2920
2002    2803
2003    2775
2004    2835
2005    2874
2006    2862
2007    2776
2008    2715
2009    2705
2010    2665
2011    2663
2012    2646
2013    2590
2014    2562
2015    2606
2016    2830
2017    2784
2018    2787
2019    2658
2020    1359
2021    2608
2022    2731
2023    2802
2024    2950
2025    2647


## 3. Cechy dynamiczne na 2001-2025 (z rozgrzewką: sezon 2000)
`add_dynamic_features(span, historical)` liczy formy, H2H, surface form, statystyki serwisu itd. dla
każdego meczu w `span`, korzystając z historii (rozgrzewka 2000 + wcześniejsze mecze span). Funkcja
radzi sobie nawet z małą rozgrzewką. To najdłuższy krok obliczeniowy notebooka.

In [ ]:
span_feat = add_dynamic_features(span, historical)   # funkcja z baseline (ns), nie z modulu m
print(f"Cechy dynamiczne policzone dla {len(span_feat)} meczów.")

# Podglad kilku kolumn cech dla najwczesniejszych meczow treningowych
sample_cols = [c for c in ["season", "winner_name", "loser_name",
                           "w_form", "l_form", "w_surface_form", "l_surface_form"]
               if c in span_feat.columns]
print(f"\nPrzykładowe cechy (pierwsze mecze {TRAIN_START}):")
print(span_feat[span_feat["season"] == TRAIN_START][sample_cols].head(5).to_string(index=False))

Cechy dynamiczne policzone dla 67153 meczów.

Przykładowe cechy (pierwsze mecze 2001):
 season        winner_name         loser_name   w_form  l_form  w_surface_form  l_surface_form
   2001     Lleyton Hewitt      Wayne Arthurs 0.700000     0.5        0.700000        0.500000
   2001 Yevgeny Kafelnikov   Francisco Clavet 0.500000     0.2        0.600000        0.200000
   2001        Taylor Dent      Magnus Norman 0.222222     0.4        0.250000        0.500000
   2001         Bjorn Phau    Todd Woodbridge 0.000000     0.4        0.000000        0.333333
   2001    Fredrik Jonsson Davide Sanguinetti 0.100000     0.5        0.333333        0.600000


## 4. Label encoding (fit tylko na treningu) + split po sezonie
`surface` i `tourney_level` kodujemy `LabelEncoder`-em fitowanym **wyłącznie** na meczach treningowych
(`season < 2024`). Nieznane kategorie w val/test mapujemy bezpiecznie na pierwszą znaną klasę (zero
wglądu w przyszłość). Potem dzielimy `span_feat` na trzy roczniki i nadajemy `match_id`.

In [ ]:
train_mask = span_feat["season"] < VAL_YEAR
le_surface, le_level = LabelEncoder(), LabelEncoder()
le_surface.fit(span_feat.loc[train_mask, "surface"])
le_level.fit(span_feat.loc[train_mask, "tourney_level"].astype(str))

def safe_transform(le, series):
    known = set(le.classes_)
    s = series.astype(str).where(series.astype(str).isin(known), le.classes_[0])
    return le.transform(s)

span_feat["surface_encoded"] = safe_transform(le_surface, span_feat["surface"])
span_feat["tourney_level_encoded"] = safe_transform(le_level, span_feat["tourney_level"].astype(str))

train_raw = span_feat[span_feat["season"] < VAL_YEAR].reset_index(drop=True)
val_raw   = span_feat[span_feat["season"] == VAL_YEAR].reset_index(drop=True)
test_raw  = span_feat[span_feat["season"] == TEST_YEAR].reset_index(drop=True)
for frame in (train_raw, val_raw, test_raw):
    frame["match_id"] = range(len(frame))

print(f"surface classes (fit na treningu): {list(le_surface.classes_)}")
print(f"tourney_level classes (fit na treningu): {list(le_level.classes_)}")
print(f"\nMecze: train={len(train_raw)} ({TRAIN_START}-{VAL_YEAR-1})"
      f"  val={len(val_raw)} ({VAL_YEAR})  test={len(test_raw)} ({TEST_YEAR})")

surface classes (fit na treningu): ['Carpet', 'Clay', 'Grass', 'Hard']
tourney_level classes (fit na treningu): ['250', '500', 'A', 'D', 'F', 'G', 'M']

Mecze: train=61556 (2001-2023)  val=2950 (2024)  test=2647 (2025)


## 5. Symetryzacja -- ten sam matrix dla wszystkich modeli
Każdy mecz zapisujemy z **dwóch** perspektyw (p1=zwycięzca / p1=przegrany), eliminując arbitralny
labeling. Wersja `shuffle=False` (chronologiczna) służy do CV w `TimeSeriesSplit`, a `shuffle=True`
do finalnego fitu. Macierz `X_*[features]` jest **identyczna** dla RF, HGB i XGBoost -- jedyna różnica
między modelami to algorytm.

In [ ]:
train_cv  = symmetrize_data(train_raw, shuffle=False)
train_fit = symmetrize_data(train_raw, shuffle=True)
val_data  = symmetrize_data(val_raw, shuffle=True)
test_data = symmetrize_data(test_raw, shuffle=True)

X_tr_cv,  y_tr_cv  = train_cv[features],  train_cv["y"]
X_tr_fit, y_tr_fit = train_fit[features], train_fit["y"]
X_val,    y_val    = val_data[features],  val_data["y"]
X_test,   y_test   = test_data[features], test_data["y"]

print(f"Próbki treningowe po symetryzacji: {len(X_tr_fit)} (2x meczów treningowych)")
print(f"Próbki val: {len(X_val)}   |   próbki test: {len(X_test)}")

# Sanity-check antysymetrii: ten sam mecz z dwoch perspektyw ma y=1 i y=0
mid = test_data["match_id"].iloc[0]
print("\nTen sam mecz widziany z 2 perspektyw (kolumna y):")
print(test_data[test_data["match_id"] == mid][["match_id", "y", "p1_name", "p2_name"]].to_string(index=False))

Próbki treningowe po symetryzacji: 123112 (2x meczów treningowych)
Próbki val: 5900   |   próbki test: 5294

Ten sam mecz widziany z 2 perspektyw (kolumna y):
 match_id  y          p1_name          p2_name
      211  1  Corentin Moutet Mitchell Krueger
      211  0 Mitchell Krueger  Corentin Moutet


## 6. [1/3] Random Forest -- tuning + ewaluacja (baseline odniesienia)
`tune_and_eval` robi `RandomizedSearchCV` na chronologicznym CV (`neg_log_loss`), refituje najlepszy
model na pełnym treningu i zwraca match accuracy (val/test) oraz metryki kalibracji. RF jest naszym
punktem odniesienia -- to z nim porównujemy boosting.

In [ ]:
rf_param_dist = {
    "n_estimators": [100, 200], "max_depth": [10, 20, None],
    "min_samples_leaf": [2, 5, 10], "max_features": ["sqrt", "log2"],
    "max_samples": [0.8, 1.0],
}
print("[1/3] Random Forest -- tuning (TimeSeriesSplit, neg_log_loss)...")
res_rf = m.tune_and_eval(
    "RandomForest",
    RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE),
    rf_param_dist, 8,
    X_tr_cv, y_tr_cv, X_tr_fit, y_tr_fit, X_val, y_val, val_data,
    X_test, y_test, test_data, compute_symmetric_match_evaluation, evaluate_calibration_quality)

print(f"  val_match ={res_rf['val_match']:.4f}   test_match={res_rf['test_match']:.4f}")
print(f"  Brier={res_rf['brier']:.4f}  logloss={res_rf['logloss']:.4f}  ECE={res_rf['ece']:.4f}")
print(f"  best_params: {res_rf['best_params']}")

[1/3] Random Forest -- tuning (TimeSeriesSplit, neg_log_loss)...


  val_match =0.6461   test_match=0.6460
  Brier=0.2182  logloss=0.6248  ECE=0.0136
  best_params: {'n_estimators': 100, 'min_samples_leaf': 10, 'max_samples': 0.8, 'max_features': 'sqrt', 'max_depth': 10}


## 7. [2/3] HistGradientBoosting -- ten sam matrix, więcej danych
Teraz boosting. Jeśli "więcej danych" miało dać przewagę boostingowi, to właśnie tutaj (~128k próbek)
powinno być widać. Te same dane, ta sama metryka, inny algorytm.

In [ ]:
hgb_param_dist = {
    "learning_rate": [0.03, 0.05, 0.1], "max_iter": [300, 500, 800],
    "max_leaf_nodes": [31, 63], "min_samples_leaf": [20, 50, 100],
    "l2_regularization": [0.0, 0.1, 1.0], "max_features": [0.6, 0.8, 1.0],
}
print("[2/3] HistGradientBoosting -- tuning...")
res_hgb = m.tune_and_eval(
    "HistGradBoost",
    HistGradientBoostingClassifier(random_state=RANDOM_STATE, early_stopping=False),
    hgb_param_dist, 12,
    X_tr_cv, y_tr_cv, X_tr_fit, y_tr_fit, X_val, y_val, val_data,
    X_test, y_test, test_data, compute_symmetric_match_evaluation, evaluate_calibration_quality)

print(f"  val_match ={res_hgb['val_match']:.4f}   test_match={res_hgb['test_match']:.4f}")
print(f"  Brier={res_hgb['brier']:.4f}  logloss={res_hgb['logloss']:.4f}  ECE={res_hgb['ece']:.4f}")
print(f"  best_params: {res_hgb['best_params']}")

[2/3] HistGradientBoosting -- tuning...


  val_match =0.6498   test_match=0.6411
  Brier=0.2172  logloss=0.6224  ECE=0.0277
  best_params: {'min_samples_leaf': 50, 'max_leaf_nodes': 63, 'max_iter': 300, 'max_features': 1.0, 'learning_rate': 0.03, 'l2_regularization': 1.0}


## 8. [3/3] XGBoost -- trzeci zawodnik na tej samej macierzy
Drugi boosting (histogramowy, ale inna implementacja i regularyzacja). Pomijany automatycznie, gdyby
biblioteka była niedostępna -- w tym środowisku jest, więc liczymy pełny tuning.

In [ ]:
results = [res_rf, res_hgb]
if m.HAS_XGB:
    from xgboost import XGBClassifier
    xgb_param_dist = {
        "n_estimators": [300, 500, 800], "max_depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.7, 0.9],
        "colsample_bytree": [0.7, 0.9], "min_child_weight": [1, 5, 10],
    }
    print("[3/3] XGBoost -- tuning...")
    res_xgb = m.tune_and_eval(
        "XGBoost",
        XGBClassifier(tree_method="hist", objective="binary:logistic",
                      eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE),
        xgb_param_dist, 12,
        X_tr_cv, y_tr_cv, X_tr_fit, y_tr_fit, X_val, y_val, val_data,
        X_test, y_test, test_data, compute_symmetric_match_evaluation, evaluate_calibration_quality)
    results.append(res_xgb)
    print(f"  val_match ={res_xgb['val_match']:.4f}   test_match={res_xgb['test_match']:.4f}")
    print(f"  Brier={res_xgb['brier']:.4f}  logloss={res_xgb['logloss']:.4f}  ECE={res_xgb['ece']:.4f}")
    print(f"  best_params: {res_xgb['best_params']}")
else:
    print("[3/3] XGBoost POMINIĘTY (brak biblioteki).")

[3/3] XGBoost -- tuning...


  val_match =0.6447   test_match=0.6490
  Brier=0.2165  logloss=0.6207  ECE=0.0189
  best_params: {'subsample': 0.9, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.7}


## 9. Tabela porównawcza + delty vs Random Forest
Zestawiamy trzy modele: match accuracy na val i test (cały sezon 2025) oraz jakość kalibracji
(Brier, log-loss, ECE). Delty liczymy względem RF -- chcemy zobaczyć, czy którykolwiek boosting
**pobił RF na accuracy**, i czy lepsza kalibracja (jeśli jest) przekłada się na cokolwiek poza
jakością prawdopodobieństw.

In [ ]:
comp = pd.DataFrame([{
    "model": r["name"], "val_match": r["val_match"], "test_match": r["test_match"],
    "Brier": r["brier"], "logloss": r["logloss"], "ECE": r["ece"],
} for r in results])
print("=" * 78)
print(f"WIELO-SEZONOWY TRENING ({TRAIN_START}-{VAL_YEAR-1}, ~{len(X_tr_fit)} próbek) | test {TEST_YEAR}")
print("=" * 78)
print(comp.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

rf = next(r for r in results if r["name"] == "RandomForest")
print("\nDelty vs Random Forest:")
for r in results:
    if r["name"] != "RandomForest":
        print(f"  {r['name']:<14} test_match={r['test_match']-rf['test_match']:+.4f}   "
              f"Brier={r['brier']-rf['brier']:+.4f}   logloss={r['logloss']-rf['logloss']:+.4f}")
print(f"\nUWAGA: test = cały sezon {TEST_YEAR} (~{len(test_raw)} meczów). CI ~ +/-2 p.p.")

WIELO-SEZONOWY TRENING (2001-2023, ~123112 próbek) | test 2025
        model  val_match  test_match  Brier  logloss    ECE
 RandomForest     0.6461      0.6460 0.2182   0.6248 0.0136
HistGradBoost     0.6498      0.6411 0.2172   0.6224 0.0277
      XGBoost     0.6447      0.6490 0.2165   0.6207 0.0189

Delty vs Random Forest:
  HistGradBoost  test_match=-0.0049   Brier=-0.0010   logloss=-0.0023
  XGBoost        test_match=+0.0030   Brier=-0.0017   logloss=-0.0041

UWAGA: test = cały sezon 2025 (~2647 meczów). CI ~ +/-2 p.p.


## Wnioski
Na największym zbiorze (trening 2001–2023, ~123 tys. próbek, test na całym sezonie 2025) wszystkie trzy algorytmy dają ~65% i różnią się tylko w granicach szumu:

| model | match accuracy 2025 | Brier |
|---|---|---|
| XGBoost | 0,6490 | 0,2165 |
| Random Forest | 0,6460 | 0,2182 |
| HistGradientBoosting | 0,6411 | 0,2172 |

Rozpiętość to 0,8 p.p., a przy 2647 meczach przedział ufności wynosi około ±2 p.p., więc nikt nie wygrywa wiarygodnie. XGBoost jest minimalnie z przodu — i na trafności, i na kalibracji — a HistGradientBoosting najsłabszy. Pasuje to do intuicji, że boosting rozwija się przy większej ilości danych: na ~72 tys. próbek Random Forest i XGBoost remisowały, a na ~123 tys. XGBoost lekko wyprzedza. Ale to wciąż szum, nie realna przewaga.

Najważniejsze jest to, że zwiększenie zbioru jakieś 36 razy nie ruszyło sufitu ~65% dla żadnego algorytmu — czyli ogranicza nas charakter cech i samego problemu, a nie wybór modelu ani ilość danych. Random Forest zostawiam jako model główny z praktycznych powodów: jest stabilny, nie wymaga dodatkowej biblioteki i wszędzie jest blisko najlepszego.